# Load Packages and Data

In [ ]:
import os
import sys
sys.path.append(os.getcwd())
sys.path.append(f"{os.getcwd()}/..")

from gp import GaussianProcess
import pandas as pd
import jax
import jax.numpy as jnp
import jax.random as jr
import jax.scipy as jsp
import numpy as np

import time
import matplotlib.pyplot as plt
import dill

In [ ]:
# y_train = jnp.load('../data/y_train.npy')

# X_train = jnp.load('../data/X_train.npy')
# y_train = y_train

# obs_count = jnp.load('../data/obs_count.npy')

X_train = jnp.load('../data/geo_X_train.npy')
y_train = jnp.load('../data/geo_y_train.npy')
obs_count = jnp.ones(X_train.shape[0]) * 3

# Configure and Run GP

In [ ]:
indoor_mask = X_train[:,2].astype(bool)
outdoor_mask = ~X_train[:,2].astype(bool)

indoor_mean = y_train[indoor_mask].mean()
outdoor_mean = y_train[outdoor_mask].mean()

def m(obs):
    return obs[2] * indoor_mean + (1 - obs[2]) * outdoor_mean


ls_xy, ls_z, os_xyz, ls_t, os_t = [0.05, 0.1, 60.0, 0.05, 30.0]
ls_xyz = jnp.array([ls_xy, ls_xy, ls_z])

def K(obs1, obs2):
    d_xyz = (obs1[:3] - obs2[:3])**2
    d_t = (obs1[3] - obs2[3])**2
    return os_xyz * jnp.exp(- (d_xyz / (2*ls_xyz**2)).sum()) + \
        os_t * jnp.exp(- d_t / (2*ls_t**2))

In [ ]:
GP = GaussianProcess(m, K)
GP.fit(X_train, y_train, obs_count)

In [ ]:
start = time.time()
chain = GP.gibbs(chains=2, samples=111)
end = time.time()
print(f"Elapsed: {end - start}")

In [ ]:
GP.variances

In [ ]:
X_val = jnp.load('../data/geo_X_val.npy')
y_val = jnp.load('../data/geo_y_val.npy')

In [ ]:
means = GP.predict(X_val, chain[1][:,10:], method='sequential')[0]
y_hat = means.mean(axis=0)

In [ ]:
sq_err = (y_val - y_hat)**2

In [ ]:
sq_err.mean()

In [ ]:
indoor_vars = chain[1][:,:,1]
outdoor_vars = chain[1][:,:,0]

In [ ]:
plt.plot(range(indoor_vars.shape[1]), indoor_vars[0])
plt.plot(range(outdoor_vars.shape[1]), outdoor_vars[0])
plt.show()

In [ ]:
cov_chains = chain[1][:,10::,:]
cov_chains.shape

# Predict on New Data

In [ ]:
X_new = jnp.load('../data/X_test.npy')
X_new = X_new.at[:,3].set(100)
with open('../data/osm_context.pkl', 'rb') as f:
    osm_context = dill.load(f)
wlon_train, wlat_train = np.load('../data/world_train.npy').T
wlon_test, wlat_test = np.load('../data/world_test.npy').T

In [ ]:
new_means, new_vars = GP.predict(X_new, cov_chains, method='sequential')

In [ ]:
vmax = jnp.quantile(y_train, 0.975)
vmin = jnp.quantile(y_train, 0.025)

base_fig, base_ax, osm_metadata = osm_context.generate_base_axis(draw_buildings=False)

sc = base_ax.scatter(wlon_test, wlat_test, c=new_means.mean(axis=0), cmap="RdYlGn", vmin=vmin, vmax=vmax)

tr = base_ax.scatter(wlon_train, wlat_train, c=y_train, cmap="RdYlGn",
                vmin=vmin, vmax=vmax, alpha=1)
plt.ticklabel_format(style='plain', axis='both', useOffset=False)

plt.colorbar(tr)
plt.tight_layout()

# plt.savefig('gridsearch1_best_plot.png', dpi=400)